# [common configs, imports etc]

In [74]:
import numpy as np

# regression MLP

## "simplest possible" regression MLP - Net

In [ ]:
class Net:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = self.X @ self.W1  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = self.Y1 @ self.W2  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # 1/(2N) * Sum (y-y_true)^2, mean over batch
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1


nn = Net(x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
W2: float64 (16, 2)


## regression MLP with bias - NetB

In [ ]:
# `Net` + CLASSIC explicit bias: every layer keeps a separate bias VECTOR (b1, b2) and
# computes  Z = X @ W + b.  This is the textbook formulation.
#   - Contrast with `Net`, which is meant to be used with the "augmentation trick"
#     instead: the caller appends a column of ones to the input and the bias lives in an
#     extra weight row -- mathematically equivalent, with no explicit b parameter.
#   - Bias is added at EVERY layer (both hidden and output) -- see the chat note on when
#     per-layer bias is/ isn't worthwhile.
class NetB:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(y1_sz)  # b1 ~ (y1_sz,)   (classic bias, zero-init)
        print(f"b1: {self.b1.dtype} {self.b1.shape}")

        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(y2_sz)  # b2 ~ (y2_sz,)   (classic bias, zero-init)
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # linear / identity
        return A

    @staticmethod
    def _df2(A):  # d(linear) == 1
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = (
            self.X @ self.W1 + self.b1
        )  # Z1 ~ (b_sz, y1_sz)   (+ b1 broadcasts over batch)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = (
            self.Y1 @ self.W2 + self.b2
        )  # Z2 ~ (b_sz, y2_sz)   (+ b2 broadcasts over batch)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # 1/(2N) * Sum (y-y_true)^2, mean over batch
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2
        # dL_db2 ~ (y2_sz,)   (bias gradient = error summed over the batch)
        self.dL_db2 = self.E2.sum(axis=0)

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1
        # dL_db1 ~ (y1_sz,)   (bias gradient = error summed over the batch)
        self.dL_db1 = self.E1.sum(axis=0)

        self.W2 += -lr * self.dL_dW2
        self.b2 += -lr * self.dL_db2
        self.W1 += -lr * self.dL_dW1
        self.b1 += -lr * self.dL_db1


nn_b = NetB(x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)


## fixed-memory regression MLP - NetFixedMem

In [ ]:
# Fixed-memory version of `Net`: ALL arrays are allocated ONCE in __init__, and the
# forward/backward hot path only writes into those preallocated buffers in place
# (no per-pass allocation). This mirrors how you'd write it in plain C: malloc every
# matrix up front, then each pass is just BLAS gemm calls with a preallocated output
# plus elementwise loops.
#
# Consequences / how it differs from Net:
#   - The batch size is FIXED at construction (buffers are sized (b_sz, ...)). Inputs
#     must be exactly (b_sz, x_sz); `forward` copies them into the preallocated X buffer.
#   - The big matmuls write into preallocated buffers via np.dot(A, B, out=C),
#     i.e. C = A @ B (a BLAS gemm into fixed memory, the C analogue).
#   - Activations are GENERIC, swappable AND allocation-free: each of _f1/_df1/_f2/_df2
#     takes an optional `out` buffer -- THE blueprint pattern, since `out` is just a
#     destination pointer in C. Three call modes from one signature:
#         f(A)           -> returns a NEW array            (simple/educational, allocates)
#         f(A, out=dst)  -> writes into dst, returns dst   (no alloc; dst may differ from A)
#         f(A, out=A)    -> modifies A IN PLACE            (no alloc, no extra buffer)
#     forward writes activations straight into Y1/Y2; backward writes activation
#     derivatives into the scratch buffers D1/D2. So the only thing C would still need
#     to add is manual malloc of these same buffers up front (already done in __init__).
#   - The SGD step scales the gradient buffers in place by lr (a "fused" update), so
#     after backward() dL_dW* hold lr*grad rather than the raw grad -- fine for training,
#     and it avoids allocating a `lr * grad` temporary.
#   - `_loss` (diagnostic) and `_dloss` still return a fresh array; giving them the same
#     `out=` treatment is a trivial further exercise.
class NetFixedMem:
    def __init__(self, *, b_sz, x_sz, y1_sz, y2_sz):
        # --- parameters (weights) ---
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

        # --- preallocated forward buffers ---
        # X ~ (b_sz, x_sz)
        self.X = np.zeros((b_sz, x_sz))  # input copied here each forward
        self.Z1 = np.zeros((b_sz, y1_sz))  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = np.zeros((b_sz, y1_sz))  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = np.zeros((b_sz, y2_sz))  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = np.zeros((b_sz, y2_sz))  # Y2 ~ (b_sz, y2_sz)

        # --- preallocated backward buffers ---
        # E2 ~ (b_sz, y2_sz)
        self.E2 = np.zeros((b_sz, y2_sz))  # dL/dZ2
        # E1 ~ (b_sz, y1_sz)
        self.E1 = np.zeros((b_sz, y1_sz))  # dL/dZ1
        # D1, D2: scratch for activation derivatives f1'(Z1), f2'(Z2)
        self.D1 = np.zeros((b_sz, y1_sz))  # D1 ~ (b_sz, y1_sz)
        self.D2 = np.zeros((b_sz, y2_sz))  # D2 ~ (b_sz, y2_sz)
        self.dL_dW2 = np.zeros((y1_sz, y2_sz))  # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW1 = np.zeros((x_sz, y1_sz))  # dL_dW1 ~ (x_sz, y1_sz)
        for name in (
            "X",
            "Z1",
            "Y1",
            "Z2",
            "Y2",
            "E2",
            "E1",
            "D1",
            "D2",
            "dL_dW1",
            "dL_dW2",
        ):
            a = getattr(self, name)
            print(f"{name}: {a.dtype} {a.shape}")

    # --- generic, swappable, allocation-free activations ---
    # Each takes an optional `out` buffer (== a destination pointer in C):
    #   out=None  -> return a NEW array      out=dst -> write into dst      out=A -> in place
    @staticmethod
    def _f1(A, out=None):  # ReLU
        return np.maximum(A, 0.0, out=out)

    @staticmethod
    def _df1(A, out=None):  # dReLU: 1.0 where A > 0 else 0.0
        # heaviside(x, h) = step fn:
        #   0 if x<0,
        #   h if x==0,
        #   1 if x>0
        # (here h=0 -> f'(0)=0)
        return np.heaviside(A, 0.0, out=out)

    @staticmethod
    def _f2(A, out=None):  # linear / identity
        if out is None:
            return A
        np.copyto(out, A)
        return out

    @staticmethod
    def _df2(A, out=None):  # d(linear) == 1
        if out is None:
            return np.ones_like(A)
        out[:] = 1.0
        return out

    @staticmethod
    def _loss(Y_predicted, Y_true):  # diagnostic only (allocates a small temp)
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def forward(self, X):
        # X ~ (b_sz, x_sz)
        np.copyto(self.X, X)  # X buffer <- input (fixed batch)
        # Z1 ~ (b_sz, y1_sz)
        np.dot(self.X, self.W1, out=self.Z1)  # Z1 = X @ W1   (preallocated out)
        # Y1 ~ (b_sz, y1_sz)
        self._f1(self.Z1, out=self.Y1)  # Y1 = f1(Z1)   (alloc-free; Z1 preserved)
        # Z2 ~ (b_sz, y2_sz)
        np.dot(self.Y1, self.W2, out=self.Z2)  # Z2 = Y1 @ W2  (preallocated out)
        # Y2 ~ (b_sz, y2_sz)
        self._f2(self.Z2, out=self.Y2)  # Y2 = f2(Z2)   (alloc-free)
        return self.Y2

    def backward(self, Y_true, lr=1e-3):
        # E2 = dL/dZ2 = dloss(Y2, Y_true) * f2'(Z2)   (generic, identical to Net)
        # E2 ~ (b_sz, y2_sz)
        self.E2[:] = self._dloss(
            self.Y2, Y_true
        )  # E2 = dloss(Y2, Y_true)  (small temp)
        self._df2(self.Z2, out=self.D2)  # D2 = f2'(Z2)  (alloc-free into scratch)
        self.E2 *= self.D2  # E2 *= f2'(Z2)

        # E1 = (E2 @ W2.T) * f1'(Z1)
        # E1 ~ (b_sz, y1_sz)
        np.dot(self.E2, self.W2.T, out=self.E1)  # E1 = E2 @ W2.T  (preallocated out)
        self._df1(self.Z1, out=self.D1)  # D1 = f1'(Z1)  (alloc-free into scratch)
        self.E1 *= self.D1  # E1 *= f1'(Z1)

        # weight gradients (preallocated out)
        # dL_dW2 ~ (y1_sz, y2_sz)
        np.dot(self.Y1.T, self.E2, out=self.dL_dW2)  # dL_dW2 = Y1.T @ E2
        # dL_dW1 ~ (x_sz, y1_sz)
        np.dot(self.X.T, self.E1, out=self.dL_dW1)  # dL_dW1 = X.T  @ E1

        # fused SGD step: scale grads in place by lr, then subtract (no temp alloc)
        self.dL_dW2 *= lr
        self.dL_dW1 *= lr
        self.W2 -= self.dL_dW2
        self.W1 -= self.dL_dW1


nn_fm = NetFixedMem(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
W2: float64 (16, 2)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)


## fixed-memory regression MLP with bias - NetFixedMemB

In [ ]:
# `NetFixedMem` + CLASSIC explicit bias (the same bias that NetB adds to Net).
# This is intentionally a near-verbatim copy of NetFixedMem; EVERY bias-specific line is
# marked with a `(bias)` comment so the diff is obvious. The fixed-memory rationale
# (preallocate once, write in place, np.dot(out=), alloc-free activations, fused SGD)
# is unchanged -- see NetFixedMem for those details.
#
# Bias additions, all staying within the preallocate-once discipline:
#   - b1, b2: bias VECTORS, allocated once (zero-init).
#   - dL_db1, dL_db2: their gradient buffers, allocated once.
#   - forward: after each matmul, add the bias in place (Z += b broadcasts over batch).
#   - backward: bias gradient = error summed over the batch, written into its buffer via
#     np.sum(..., out=...); then folded into the same fused SGD step as the weights.
class NetFixedMemB:
    def __init__(self, *, b_sz, x_sz, y1_sz, y2_sz):
        # --- parameters (weights + biases) ---
        self.W1 = np.random.randn(x_sz, y1_sz)  # W1 ~ (x_sz, y1_sz)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(
            y1_sz
        )  # (bias) b1 ~ (y1_sz,)   classic bias vector, zero-init
        print(f"b1: {self.b1.dtype} {self.b1.shape}")
        self.W2 = np.random.randn(y1_sz, y2_sz)  # W2 ~ (y1_sz, y2_sz)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(
            y2_sz
        )  # (bias) b2 ~ (y2_sz,)   classic bias vector, zero-init
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

        # --- preallocated forward buffers ---
        # X ~ (b_sz, x_sz)
        self.X = np.zeros((b_sz, x_sz))  # input copied here each forward
        self.Z1 = np.zeros((b_sz, y1_sz))  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = np.zeros((b_sz, y1_sz))  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = np.zeros((b_sz, y2_sz))  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = np.zeros((b_sz, y2_sz))  # Y2 ~ (b_sz, y2_sz)

        # --- preallocated backward buffers ---
        # E2 ~ (b_sz, y2_sz)
        self.E2 = np.zeros((b_sz, y2_sz))  # dL/dZ2
        # E1 ~ (b_sz, y1_sz)
        self.E1 = np.zeros((b_sz, y1_sz))  # dL/dZ1
        # D1, D2: scratch for activation derivatives f1'(Z1), f2'(Z2)
        self.D1 = np.zeros((b_sz, y1_sz))  # D1 ~ (b_sz, y1_sz)
        self.D2 = np.zeros((b_sz, y2_sz))  # D2 ~ (b_sz, y2_sz)
        self.dL_dW2 = np.zeros((y1_sz, y2_sz))  # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW1 = np.zeros((x_sz, y1_sz))  # dL_dW1 ~ (x_sz, y1_sz)
        # (bias) preallocated bias-gradient buffers
        self.dL_db2 = np.zeros(y2_sz)  # (bias) dL_db2 ~ (y2_sz,)
        self.dL_db1 = np.zeros(y1_sz)  # (bias) dL_db1 ~ (y1_sz,)
        for name in (
            "X",
            "Z1",
            "Y1",
            "Z2",
            "Y2",
            "E2",
            "E1",
            "D1",
            "D2",
            "dL_dW1",
            "dL_dW2",
            "dL_db1",  # (bias)
            "dL_db2",  # (bias)
        ):
            a = getattr(self, name)
            print(f"{name}: {a.dtype} {a.shape}")

    # --- generic, swappable, allocation-free activations ---
    # Each takes an optional `out` buffer (== a destination pointer in C):
    #   out=None  -> return a NEW array      out=dst -> write into dst      out=A -> in place
    @staticmethod
    def _f1(A, out=None):  # ReLU
        return np.maximum(A, 0.0, out=out)

    @staticmethod
    def _df1(A, out=None):  # dReLU: 1.0 where A > 0 else 0.0
        # heaviside(x, h) = step fn:
        #   0 if x<0,
        #   h if x==0,
        #   1 if x>0
        # (here h=0 -> f'(0)=0)
        return np.heaviside(A, 0.0, out=out)

    @staticmethod
    def _f2(A, out=None):  # linear / identity
        if out is None:
            return A
        np.copyto(out, A)
        return out

    @staticmethod
    def _df2(A, out=None):  # d(linear) == 1
        if out is None:
            return np.ones_like(A)
        out[:] = 1.0
        return out

    @staticmethod
    def _loss(Y_predicted, Y_true):  # diagnostic only (allocates a small temp)
        return 0.5 * ((Y_predicted - Y_true) ** 2).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def forward(self, X):
        # X ~ (b_sz, x_sz)
        np.copyto(self.X, X)  # X buffer <- input (fixed batch)
        # Z1 ~ (b_sz, y1_sz)
        np.dot(self.X, self.W1, out=self.Z1)  # Z1 = X @ W1   (preallocated out)
        self.Z1 += self.b1  # (bias) Z1 += b1   (in-place, broadcasts over batch)
        # Y1 ~ (b_sz, y1_sz)
        self._f1(self.Z1, out=self.Y1)  # Y1 = f1(Z1)   (alloc-free; Z1 preserved)
        # Z2 ~ (b_sz, y2_sz)
        np.dot(self.Y1, self.W2, out=self.Z2)  # Z2 = Y1 @ W2  (preallocated out)
        self.Z2 += self.b2  # (bias) Z2 += b2   (in-place, broadcasts over batch)
        # Y2 ~ (b_sz, y2_sz)
        self._f2(self.Z2, out=self.Y2)  # Y2 = f2(Z2)   (alloc-free)
        return self.Y2

    def backward(self, Y_true, lr=1e-3):
        # E2 = dL/dZ2 = dloss(Y2, Y_true) * f2'(Z2)   (generic, identical to Net)
        # E2 ~ (b_sz, y2_sz)
        self.E2[:] = self._dloss(
            self.Y2, Y_true
        )  # E2 = dloss(Y2, Y_true)  (small temp)
        self._df2(self.Z2, out=self.D2)  # D2 = f2'(Z2)  (alloc-free into scratch)
        self.E2 *= self.D2  # E2 *= f2'(Z2)

        # E1 = (E2 @ W2.T) * f1'(Z1)
        # E1 ~ (b_sz, y1_sz)
        np.dot(self.E2, self.W2.T, out=self.E1)  # E1 = E2 @ W2.T  (preallocated out)
        self._df1(self.Z1, out=self.D1)  # D1 = f1'(Z1)  (alloc-free into scratch)
        self.E1 *= self.D1  # E1 *= f1'(Z1)

        # weight gradients (preallocated out)
        # dL_dW2 ~ (y1_sz, y2_sz)
        np.dot(self.Y1.T, self.E2, out=self.dL_dW2)  # dL_dW2 = Y1.T @ E2
        # (bias) dL_db2 = E2 summed over the batch  (np.sum into preallocated buffer)
        np.sum(self.E2, axis=0, out=self.dL_db2)
        # dL_dW1 ~ (x_sz, y1_sz)
        np.dot(self.X.T, self.E1, out=self.dL_dW1)  # dL_dW1 = X.T  @ E1
        # (bias) dL_db1 = E1 summed over the batch
        np.sum(self.E1, axis=0, out=self.dL_db1)

        # fused SGD step: scale grads in place by lr, then subtract (no temp alloc)
        self.dL_dW2 *= lr
        self.dL_dW1 *= lr
        self.dL_db2 *= lr  # (bias)
        self.dL_db1 *= lr  # (bias)
        self.W2 -= self.dL_dW2
        self.W1 -= self.dL_dW1
        self.b2 -= self.dL_db2  # (bias)
        self.b1 -= self.dL_db1  # (bias)


nn_fm_b = NetFixedMemB(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)
dL_db1: float64 (16,)
dL_db2: float64 (2,)


## canonical PyTorch regression MLP - NetTorch

In [ ]:
# Canonical PyTorch equivalent of NetB (regression MLP with classic bias):
#   Linear(x_sz->y1_sz) -> ReLU -> Linear(y1_sz->y2_sz), linear output for regression.
# Idiomatic torch (nn.Module + autograd): no hand-written backward, and forward() returns the
# raw predictions; you'd train it with nn.MSELoss + an optimizer (same recipe as the torch
# training loop in the task section, swapping CrossEntropyLoss for MSELoss).
# This is the regression sibling of DigitsRecognizerPytorch (which is the NetB-classifier).
import torch
import torch.nn as nn
import torch.nn.functional as F


class NetTorch(nn.Module):
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        super().__init__()
        self.fc1 = nn.Linear(x_sz, y1_sz)  # weight ~ (y1_sz, x_sz), bias ~ (y1_sz,)
        self.fc2 = nn.Linear(y1_sz, y2_sz)  # weight ~ (y2_sz, y1_sz), bias ~ (y2_sz,)

        # match NetB's init: plain randn weights (std 1) and zero biases
        # (torch's Linear default differs, so we set it explicitly)
        nn.init.normal_(self.fc1.weight, std=1.0)
        nn.init.normal_(self.fc2.weight, std=1.0)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)

        for (
            name,
            p,
        ) in self.named_parameters():  # print param shapes/dtypes like the NumPy nets
            print(f"{name}: {p.dtype} {tuple(p.shape)}")

    def forward(self, X):  # X ~ (b, x_sz) float tensor -> predictions ~ (b, y2_sz)
        return self.fc2(F.relu(self.fc1(X)))


nt = NetTorch(x_sz=5, y1_sz=16, y2_sz=2)

fc1.weight: torch.float32 (16, 5)
fc1.bias: torch.float32 (16,)
fc2.weight: torch.float32 (2, 16)
fc2.bias: torch.float32 (2,)


# task example: energy efficiency regression

In [ ]:
# End-to-end Energy-Efficiency REGRESSION test for our regression nets. Mirrors the MNIST
# section, but for regression: continuous targets, MSE loss (Net._loss), RMSE as the metric.
# `net_cls` is a parameter so the SAME routine works for any class with the Net interface
# (forward / backward; no one_hot/predict needed -- forward() IS the prediction).
#
# sklearn is used ONLY to download the dataset (the same bits a torch baseline would see);
# ALL training/inference is our NumPy code. The UCI Energy Efficiency dataset predicts a
# building's heating load (y1) and cooling load (y2) from 8 shape features -> a natural
# 2-output regression that matches Net's multi-output design.
from sklearn.datasets import fetch_openml


def load_energy(*, test_frac=0.2, seed=0):
    """Download the UCI Energy Efficiency dataset and return standardized NumPy arrays.
    fetch_openml (with BOTH targets pulled out) hands us:
      data   : float64 (768, 8)  -- 8 building-shape features V1..V8
      target : (768, 2)          -- y1 = heating load, y2 = cooling load (kWh/m^2)
    """
    ds = fetch_openml(
        name="energy-efficiency",
        as_frame=False,
        parser="liac-arff",
        target_column=["y1", "y2"],
    )
    X = ds.data.astype(np.float64)  # (768, 8) features
    Y = ds.target.astype(np.float64)  # (768, 2) heating + cooling load
    print(
        f"raw X: {X.shape} {X.dtype}   raw Y: {Y.shape} {Y.dtype}  (targets: {ds.target_names})"
    )

    # shuffle, then split into train / test
    rng = np.random.default_rng(seed)
    perm = rng.permutation(X.shape[0])
    n_test = int(X.shape[0] * test_frac)
    te_idx, tr_idx = perm[:n_test], perm[n_test:]
    X_train, Y_train, X_test, Y_test = X[tr_idx], Y[tr_idx], X[te_idx], Y[te_idx]

    # standardize features AND targets using TRAIN stats only (no test leakage). MSE + SGD
    # need a sane scale; here raw features span ~0.1 to ~670, so this step is essential.
    x_mu, x_sd = X_train.mean(0), X_train.std(0)
    y_mu, y_sd = Y_train.mean(0), Y_train.std(0)
    X_train, X_test = (X_train - x_mu) / x_sd, (X_test - x_mu) / x_sd
    Y_train, Y_test = (Y_train - y_mu) / y_sd, (Y_test - y_mu) / y_sd
    print(
        f"X_train: {X_train.shape} {X_train.dtype}   Y_train: {Y_train.shape} {Y_train.dtype}"
    )
    print(
        f"X_test : {X_test.shape} {X_test.dtype}   Y_test : {Y_test.shape} {Y_test.dtype}"
    )
    return (
        X_train,
        Y_train,
        X_test,
        Y_test,
        y_mu,
        y_sd,
    )  # y_mu/y_sd -> report in original units


def rmse(net, X, Y, y_sd):
    # forward() gives standardized predictions; rescale the per-target RMSE to kWh/m^2
    P = net.forward(X)  # (N, 2) standardized predictions
    return (
        np.sqrt(((P - Y) ** 2).mean(axis=0)) * y_sd
    )  # (2,) RMSE per target, original units


# NOTE on the modest width (y1_sz=32): NetB/NetTorch use plain std-1 weight init (randn),
# NOT He init. With standardized inputs, widening the hidden layer (e.g. y1_sz=64) blows up
# the pre-activations and training diverges to NaN. This is exactly the fragility that
# motivated He init in the DigitsRecognizer* classifiers; here we just keep the width where
# std-1 init stays stable (switching to He init would lift this limit -- left as an exercise).
def train_and_test_energy(
    net_cls, *, y1_sz=32, epochs=300, batch_size=32, lr=0.03, seed=0
):
    """Train `net_cls` on Energy Efficiency with mini-batch SGD; print shapes + per-target RMSE."""
    X_train, Y_train, X_test, Y_test, y_mu, y_sd = load_energy(seed=seed)
    n, x_sz = X_train.shape  # 615, 8
    y2_sz = Y_train.shape[1]  # 2  (heating + cooling load)

    np.random.seed(seed)  # reproducible weight init
    net = net_cls(x_sz=x_sz, y1_sz=y1_sz, y2_sz=y2_sz)  # __init__ prints param shapes

    rng = np.random.default_rng(seed)
    for epoch in range(epochs):
        order = rng.permutation(n)  # reshuffle each epoch
        for i in range(0, n, batch_size):
            idx = order[i : i + batch_size]
            Xb, Yb = X_train[idx], Y_train[idx]  # (b, 8) features, (b, 2) targets
            if epoch == 0 and i == 0:  # peek at the very first mini-batch
                print(
                    f"batch X: {Xb.shape} {Xb.dtype}   batch Y: {Yb.shape} {Yb.dtype}"
                )
            net.forward(Xb)
            net.backward(Yb, lr=lr)
        if epoch == 0 or (epoch + 1) % 50 == 0:  # RMSE = (heating, cooling), in kWh/m^2
            print(
                f"epoch {epoch + 1}/{epochs}: train RMSE = {rmse(net, X_train, Y_train, y_sd).round(3)}   test RMSE = {rmse(net, X_test, Y_test, y_sd).round(3)}"
            )

    pred = net.forward(X_test[:3]) * y_sd + y_mu  # de-standardize for a readable sample
    true = Y_test[:3] * y_sd + y_mu
    print(f"predict sample (heating, cooling kWh/m^2): {pred.round(2).tolist()}")
    print(f"      true                              : {true.round(2).tolist()}")
    return net


net_energy = train_and_test_energy(NetB)

raw X: (768, 8) float64   raw Y: (768, 2) float64  (targets: ['y1', 'y2'])
X_train: (615, 8) float64   Y_train: (615, 2) float64
X_test : (153, 8) float64   Y_test : (153, 2) float64
W1: float64 (8, 32)
b1: float64 (32,)
W2: float64 (32, 2)
b2: float64 (2,)
batch X: (32, 8) float64   batch Y: (32, 2) float64
epoch 1/300: train RMSE = [16.044 10.303]   test RMSE = [17.706 10.746]
epoch 50/300: train RMSE = [4.119 2.991]   test RMSE = [5.299 3.139]
epoch 100/300: train RMSE = [3.701 2.668]   test RMSE = [4.746 2.804]
epoch 150/300: train RMSE = [3.447 2.523]   test RMSE = [4.392 2.743]
epoch 200/300: train RMSE = [3.155 2.374]   test RMSE = [4.208 2.514]
epoch 250/300: train RMSE = [3.147 2.427]   test RMSE = [4.081 2.409]
epoch 300/300: train RMSE = [2.777 2.361]   test RMSE = [3.811 2.258]
predict sample (heating, cooling kWh/m^2): [[5.06, 7.78], [7.03, 4.97], [0.68, 5.74]]
      true                              : [[4.0, 6.0], [4.0, 4.0], [2.0, 4.0]]


In [ ]:
# Idiomatic PyTorch training loop for NetTorch -- the canonical counterpart to
# train_and_test_energy above. Same data (reuses load_energy) and same plain SGD, so the
# numbers are comparable; the only regression-specific change vs the MNIST torch loop is
# nn.MSELoss instead of nn.CrossEntropyLoss (and continuous float targets, no argmax).
from torch.utils.data import DataLoader, TensorDataset


def train_and_test_energy_torch(
    *, y1_sz=32, epochs=300, batch_size=32, lr=0.03, seed=0
):
    X_train, Y_train, X_test, Y_test, y_mu, y_sd = load_energy(seed=seed)

    # numpy -> float32 tensors (both inputs AND targets are continuous for regression)
    Xtr = torch.tensor(X_train, dtype=torch.float32)  # (615, 8)
    Ytr = torch.tensor(Y_train, dtype=torch.float32)  # (615, 2)
    Xte = torch.tensor(X_test, dtype=torch.float32)  # (153, 8)
    Yte = torch.tensor(Y_test, dtype=torch.float32)  # (153, 2)
    train_loader = DataLoader(
        TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True
    )

    torch.manual_seed(seed)  # reproducible weight init
    model = NetTorch(x_sz=Xtr.shape[1], y1_sz=y1_sz, y2_sz=Ytr.shape[1])
    criterion = (
        nn.MSELoss()
    )  # mean squared error -- the regression loss (vs CrossEntropy)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr
    )  # plain SGD, like our NumPy nets
    y_sd_t = torch.tensor(
        y_sd, dtype=torch.float32
    )  # to rescale RMSE to original units

    @torch.no_grad()
    def rmse(X, Y):  # per-target RMSE in kWh/m^2
        model.eval()
        return (((model(X) - Y) ** 2).mean(dim=0).sqrt() * y_sd_t).numpy()

    Xb0, Yb0 = next(iter(train_loader))  # peek at one mini-batch
    print(
        f"batch X: {tuple(Xb0.shape)} {Xb0.dtype}   batch Y: {tuple(Yb0.shape)} {Yb0.dtype}"
    )
    for epoch in range(epochs):
        model.train()  # train mode
        for Xb, Yb in train_loader:  # Xb ~ (b, 8), Yb ~ (b, 2) float32
            optimizer.zero_grad()  # clear grads from the previous step
            loss = criterion(model(Xb), Yb)  # forward pass -> scalar MSE
            loss.backward()  # autograd fills every parameter's .grad
            optimizer.step()  # SGD update: p -= lr * p.grad
        if epoch == 0 or (epoch + 1) % 50 == 0:  # RMSE = (heating, cooling), in kWh/m^2
            print(
                f"epoch {epoch + 1}/{epochs}: train RMSE = {rmse(Xtr, Ytr).round(3)}   test RMSE = {rmse(Xte, Yte).round(3)}"
            )

    with torch.no_grad():
        pred = (
            model(Xte[:3]).numpy() * y_sd + y_mu
        )  # de-standardize for a readable sample
    true = Y_test[:3] * y_sd + y_mu
    print(f"predict sample (heating, cooling kWh/m^2): {pred.round(2).tolist()}")
    print(f"      true                              : {true.round(2).tolist()}")
    return model


net_energy_torch = train_and_test_energy_torch()

raw X: (768, 8) float64   raw Y: (768, 2) float64  (targets: ['y1', 'y2'])
X_train: (615, 8) float64   Y_train: (615, 2) float64
X_test : (153, 8) float64   Y_test : (153, 2) float64
fc1.weight: torch.float32 (32, 8)
fc1.bias: torch.float32 (32,)
fc2.weight: torch.float32 (2, 32)
fc2.bias: torch.float32 (2,)
batch X: (32, 8) torch.float32   batch Y: (32, 2) torch.float32
epoch 1/300: train RMSE = [10.988 11.395]   test RMSE = [10.314 10.793]
epoch 50/300: train RMSE = [3.656 3.595]   test RMSE = [4.258 4.335]
epoch 100/300: train RMSE = [2.903 2.57 ]   test RMSE = [3.644 3.287]
epoch 150/300: train RMSE = [2.665 2.349]   test RMSE = [3.381 2.987]
epoch 200/300: train RMSE = [2.457 2.174]   test RMSE = [3.071 2.625]
epoch 250/300: train RMSE = [2.378 2.087]   test RMSE = [2.971 2.534]
epoch 300/300: train RMSE = [3.165 2.066]   test RMSE = [3.669 2.518]
predict sample (heating, cooling kWh/m^2): [[7.33, 8.32], [8.62, 6.15], [2.95, 5.64]]
      true                              : [[4.0, 

# classification MLP

## "simplest possible" classification MLP - DigitsRecognizerNet

In [54]:
# Closest possible adaptation of `Net` to MNIST digit classification.
#
# What CHANGES vs. Net (and why):
#   - sizes: x_sz=784 (28x28 pixels), y2_sz=10 (one score per digit class).
#   - _f2: linear -> softmax, so the 10 outputs form a probability distribution.
#   - _loss: mean squared error -> mean cross-entropy (the natural classification loss).
#   - weight init: scaled by 1/sqrt(fan_in) (He/Xavier) instead of plain randn,
#     otherwise 784 inputs make the pre-activations huge and softmax saturates.
#
# What STAYS IDENTICAL to Net (the elegant part):
#   - _f1/_df1 (ReLU), forward(), backward().
#   - _dloss returns (Y_predicted - Y_true) / N, and _df2 returns ones, EXACTLY as
#     in Net. For softmax + cross-entropy the output error dL/dZ2 collapses to
#     (softmax - one_hot)/N == (Y2 - Y_true)/N -- the same form as linear + MSE.
#     So we fold the softmax/CE derivative into this pair and reuse backward() verbatim.
#
# Y_true is expected one-hot: shape (batch, 10). Use one_hot(labels) to build it.
class DigitsRecognizerNet:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        # W1 ~ (x_sz, y1_sz)
        self.W1 = np.random.randn(x_sz, y1_sz) * np.sqrt(2.0 / x_sz)  # He init (ReLU)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")

        # W2 ~ (y1_sz, y2_sz)
        self.W2 = np.random.randn(y1_sz, y2_sz) * np.sqrt(
            1.0 / y1_sz
        )  # Xavier-ish (softmax)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # softmax (row-wise, numerically stable)
        # subtract the row-wise max for numerical stability
        # (fine bc doftmax is shift-invariant: subtracting any constant c from a row's logits gives the identical result )
        A = A - A.max(axis=1, keepdims=True)

        E = np.exp(A)

        return E / E.sum(axis=1, keepdims=True)

    @staticmethod
    def _df2(A):  # folded into _dloss for softmax+CE -> identity
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = self.X @ self.W1  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = self.Y1 @ self.W2  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # mean cross-entropy over batch
        eps = 1e-12
        return -(Y_true * np.log(Y_predicted + eps)).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):  # softmax+CE: dL/dZ2 == (Y2 - Y_true)/N
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1

        self.W2 += -lr * self.dL_dW2
        self.W1 += -lr * self.dL_dW1

    @staticmethod
    def one_hot(labels, n_classes=10):  # integer labels (batch,) -> (batch, n_classes)
        Y = np.zeros((labels.shape[0], n_classes))
        Y[np.arange(labels.shape[0]), labels] = 1.0
        return Y

    def predict(self, X):  # class index with highest probability
        return self.forward(X).argmax(axis=1)


dnn = DigitsRecognizerNet(x_sz=784, y1_sz=128, y2_sz=10)

W1: float64 (784, 128)
W2: float64 (128, 10)


## classification MLP with bias - DigitsRecognizerNetB

In [75]:
# `DigitsRecognizerNetB` = `DigitsRecognizerNet` + classic bias, exactly the same step
# `NetB` takes from `Net`: add a bias vector at every layer and learn it.
#   - forward computes  Z = X @ W + b  (b broadcasts over the batch).
#   - backward adds the bias gradient  dL_db = error summed over the batch.
#   - `DigitsRecognizerNet` (no bias) is meant for the augmentation trick (extra column of
#     ones on the input); this version carries explicit b1/b2 instead.
# Everything else (softmax _f2, mean cross-entropy _loss, He/Xavier init, one_hot, predict)
# is identical to `DigitsRecognizerNet`.
#
# Y_true is expected one-hot: shape (batch, 10). Use one_hot(labels) to build it.
class DigitsRecognizerNetB:
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        # W1 ~ (x_sz, y1_sz)
        self.W1 = np.random.randn(x_sz, y1_sz) * np.sqrt(2.0 / x_sz)  # He init (ReLU)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(y1_sz)  # b1 ~ (y1_sz,)   (classic bias, zero-init)
        print(f"b1: {self.b1.dtype} {self.b1.shape}")

        # W2 ~ (y1_sz, y2_sz)
        self.W2 = np.random.randn(y1_sz, y2_sz) * np.sqrt(
            1.0 / y1_sz
        )  # Xavier-ish (softmax)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(y2_sz)  # b2 ~ (y2_sz,)   (classic bias, zero-init)
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

    @staticmethod
    def _f1(A):  # ReLU
        return A * (A > 0)

    @staticmethod
    def _df1(A):  # dReLU
        return (A > 0).astype(A.dtype)

    @staticmethod
    def _f2(A):  # softmax (row-wise, numerically stable)
        # subtract the row-wise max for numerical stability
        # (fine bc softmax is shift-invariant: subtracting any constant c from a row's logits gives the identical result)
        A = A - A.max(axis=1, keepdims=True)

        E = np.exp(A)

        return E / E.sum(axis=1, keepdims=True)

    @staticmethod
    def _df2(A):  # folded into _dloss for softmax+CE -> identity
        return np.ones_like(A)

    def forward(self, X):
        self.X = X  # X ~ (b_sz, x_sz)
        self.Z1 = (
            self.X @ self.W1 + self.b1
        )  # Z1 ~ (b_sz, y1_sz)   (+ b1 broadcasts over batch)
        self.Y1 = self._f1(self.Z1)  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = (
            self.Y1 @ self.W2 + self.b2
        )  # Z2 ~ (b_sz, y2_sz)   (+ b2 broadcasts over batch)
        self.Y2 = self._f2(self.Z2)  # Y2 ~ (b_sz, y2_sz)
        return self.Y2

    @staticmethod
    def _loss(Y_predicted, Y_true):  # mean cross-entropy over batch
        eps = 1e-12
        return -(Y_true * np.log(Y_predicted + eps)).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):  # softmax+CE: dL/dZ2 == (Y2 - Y_true)/N
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def backward(self, Y_true, lr=1e-3):
        # E2 ~ (b_sz, y2_sz)
        self.E2 = self._dloss(self.Y2, Y_true) * self._df2(self.Z2)

        # E1 ~ (b_sz, y1_sz)
        self.E1 = self.E2 @ self.W2.T * self._df1(self.Z1)

        # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW2 = self.Y1.T @ self.E2
        # dL_db2 ~ (y2_sz,)   (bias gradient = error summed over the batch)
        self.dL_db2 = self.E2.sum(axis=0)

        # dL_dW1 ~ (x_sz, y1_sz)
        self.dL_dW1 = self.X.T @ self.E1
        # dL_db1 ~ (y1_sz,)   (bias gradient = error summed over the batch)
        self.dL_db1 = self.E1.sum(axis=0)

        self.W2 += -lr * self.dL_dW2
        self.b2 += -lr * self.dL_db2
        self.W1 += -lr * self.dL_dW1
        self.b1 += -lr * self.dL_db1

    @staticmethod
    def one_hot(labels, n_classes=10):  # integer labels (batch,) -> (batch, n_classes)
        Y = np.zeros((labels.shape[0], n_classes))
        Y[np.arange(labels.shape[0]), labels] = 1.0
        return Y

    def predict(self, X):  # class index with highest probability
        return self.forward(X).argmax(axis=1)


dnn_b = DigitsRecognizerNetB(x_sz=784, y1_sz=128, y2_sz=10)

W1: float64 (784, 128)
b1: float64 (128,)
W2: float64 (128, 10)
b2: float64 (10,)


## fixed-memory classification MLP with bias - DigitsRecognizerNetFixedMemB

In [ ]:
# `NetFixedMemB` made into a CLASSIFIER: the fixed-memory counterpart of DigitsRecognizerNetB
# (softmax + cross-entropy, classic bias), staying inside the same preallocate-once discipline
# as NetFixedMem/NetFixedMemB (no extra optimizations beyond those basics).
#
# What CHANGES vs NetFixedMemB (exactly the DigitsRecognizerNetB changes):
#   - weight init scaled by 1/sqrt(fan_in) (He/Xavier) so softmax doesn't saturate.
#   - _f2: linear -> softmax (row-wise, numerically stable). Softmax needs a couple of small
#     temporaries (max/exp/sum), so it is NOT strictly alloc-free -- fine, the activations stay
#     GENERIC and swappable; squeezing softmax into preallocated buffers is left as an exercise.
#     It still honors the out= contract (writes its result into out when one is given).
#   - _loss: mean squared error -> mean cross-entropy.
# What STAYS IDENTICAL to NetFixedMemB:
#   - _f1/_df1 (ReLU), _df2 (== ones) and _dloss (== (Y2 - Y_true)/N), so forward()/backward()
#     are reused VERBATIM: softmax+CE folds dL/dZ2 into (Y2 - Y_true)/N, the same form as
#     linear+MSE (see DigitsRecognizerNet for the derivation). Every bias line keeps its (bias) tag.
#
# Batch size is FIXED at construction (b_sz); one_hot()/predict() match DigitsRecognizerNetB,
# but predict() expects exactly b_sz rows. Y_true is one-hot: shape (b_sz, y2_sz).
class DigitsRecognizerNetFixedMemB:
    def __init__(self, *, b_sz, x_sz, y1_sz, y2_sz):
        # --- parameters (weights + biases) ---
        self.W1 = np.random.randn(x_sz, y1_sz) * np.sqrt(
            2.0 / x_sz
        )  # W1 ~ (x_sz, y1_sz)   He init (ReLU)
        print(f"W1: {self.W1.dtype} {self.W1.shape}")
        self.b1 = np.zeros(
            y1_sz
        )  # (bias) b1 ~ (y1_sz,)   classic bias vector, zero-init
        print(f"b1: {self.b1.dtype} {self.b1.shape}")
        self.W2 = np.random.randn(y1_sz, y2_sz) * np.sqrt(
            1.0 / y1_sz
        )  # W2 ~ (y1_sz, y2_sz)   Xavier-ish (softmax)
        print(f"W2: {self.W2.dtype} {self.W2.shape}")
        self.b2 = np.zeros(
            y2_sz
        )  # (bias) b2 ~ (y2_sz,)   classic bias vector, zero-init
        print(f"b2: {self.b2.dtype} {self.b2.shape}")

        # --- preallocated forward buffers ---
        # X ~ (b_sz, x_sz)
        self.X = np.zeros((b_sz, x_sz))  # input copied here each forward
        self.Z1 = np.zeros((b_sz, y1_sz))  # Z1 ~ (b_sz, y1_sz)
        self.Y1 = np.zeros((b_sz, y1_sz))  # Y1 ~ (b_sz, y1_sz)
        self.Z2 = np.zeros((b_sz, y2_sz))  # Z2 ~ (b_sz, y2_sz)
        self.Y2 = np.zeros((b_sz, y2_sz))  # Y2 ~ (b_sz, y2_sz)

        # --- preallocated backward buffers ---
        # E2 ~ (b_sz, y2_sz)
        self.E2 = np.zeros((b_sz, y2_sz))  # dL/dZ2
        # E1 ~ (b_sz, y1_sz)
        self.E1 = np.zeros((b_sz, y1_sz))  # dL/dZ1
        # D1, D2: scratch for activation derivatives f1'(Z1), f2'(Z2)
        self.D1 = np.zeros((b_sz, y1_sz))  # D1 ~ (b_sz, y1_sz)
        self.D2 = np.zeros((b_sz, y2_sz))  # D2 ~ (b_sz, y2_sz)
        self.dL_dW2 = np.zeros((y1_sz, y2_sz))  # dL_dW2 ~ (y1_sz, y2_sz)
        self.dL_dW1 = np.zeros((x_sz, y1_sz))  # dL_dW1 ~ (x_sz, y1_sz)
        # (bias) preallocated bias-gradient buffers
        self.dL_db2 = np.zeros(y2_sz)  # (bias) dL_db2 ~ (y2_sz,)
        self.dL_db1 = np.zeros(y1_sz)  # (bias) dL_db1 ~ (y1_sz,)
        for name in (
            "X",
            "Z1",
            "Y1",
            "Z2",
            "Y2",
            "E2",
            "E1",
            "D1",
            "D2",
            "dL_dW1",
            "dL_dW2",
            "dL_db1",
            "dL_db2",  # (bias) last two
        ):
            a = getattr(self, name)
            print(f"{name}: {a.dtype} {a.shape}")

    # --- generic, swappable activations (out= contract, as in NetFixedMem) ---
    @staticmethod
    def _f1(A, out=None):  # ReLU
        return np.maximum(A, 0.0, out=out)

    @staticmethod
    def _df1(A, out=None):  # dReLU: 1.0 where A > 0 else 0.0
        return np.heaviside(
            A, 0.0, out=out
        )  # heaviside(x,0): 0 if x<0, 0 if x==0, 1 if x>0

    @staticmethod
    def _f2(A, out=None):  # softmax (row-wise, numerically stable)
        S = A - A.max(axis=1, keepdims=True)  # small temp; shift-invariant -> stability
        np.exp(S, out=S)
        S /= S.sum(axis=1, keepdims=True)
        if out is None:
            return S
        np.copyto(out, S)
        return out

    @staticmethod
    def _df2(A, out=None):  # folded into _dloss for softmax+CE -> identity (== ones)
        if out is None:
            return np.ones_like(A)
        out[:] = 1.0
        return out

    @staticmethod
    def _loss(Y_predicted, Y_true):  # mean cross-entropy over batch (diagnostic only)
        eps = 1e-12
        return -(Y_true * np.log(Y_predicted + eps)).sum() / Y_predicted.shape[0]

    @staticmethod
    def _dloss(Y_predicted, Y_true):  # softmax+CE: dL/dZ2 == (Y2 - Y_true)/N
        return (Y_predicted - Y_true) / Y_predicted.shape[0]

    def forward(self, X):
        # X ~ (b_sz, x_sz)
        np.copyto(self.X, X)  # X buffer <- input (fixed batch)
        # Z1 ~ (b_sz, y1_sz)
        np.dot(self.X, self.W1, out=self.Z1)  # Z1 = X @ W1   (preallocated out)
        self.Z1 += self.b1  # (bias) Z1 += b1   (in-place, broadcasts over batch)
        # Y1 ~ (b_sz, y1_sz)
        self._f1(self.Z1, out=self.Y1)  # Y1 = f1(Z1)   (alloc-free; Z1 preserved)
        # Z2 ~ (b_sz, y2_sz)
        np.dot(self.Y1, self.W2, out=self.Z2)  # Z2 = Y1 @ W2  (preallocated out)
        self.Z2 += self.b2  # (bias) Z2 += b2   (in-place, broadcasts over batch)
        # Y2 ~ (b_sz, y2_sz)
        self._f2(self.Z2, out=self.Y2)  # Y2 = f2(Z2) = softmax(Z2)
        return self.Y2

    def backward(self, Y_true, lr=1e-3):
        # E2 = dL/dZ2 = dloss(Y2, Y_true) * f2'(Z2)   (generic; for softmax+CE f2' == 1)
        # E2 ~ (b_sz, y2_sz)
        self.E2[:] = self._dloss(
            self.Y2, Y_true
        )  # E2 = dloss(Y2, Y_true)  (small temp)
        self._df2(self.Z2, out=self.D2)  # D2 = f2'(Z2)  (alloc-free into scratch)
        self.E2 *= self.D2  # E2 *= f2'(Z2)

        # E1 = (E2 @ W2.T) * f1'(Z1)
        # E1 ~ (b_sz, y1_sz)
        np.dot(self.E2, self.W2.T, out=self.E1)  # E1 = E2 @ W2.T  (preallocated out)
        self._df1(self.Z1, out=self.D1)  # D1 = f1'(Z1)  (alloc-free into scratch)
        self.E1 *= self.D1  # E1 *= f1'(Z1)

        # weight gradients (preallocated out)
        # dL_dW2 ~ (y1_sz, y2_sz)
        np.dot(self.Y1.T, self.E2, out=self.dL_dW2)  # dL_dW2 = Y1.T @ E2
        # (bias) dL_db2 = E2 summed over the batch
        np.sum(self.E2, axis=0, out=self.dL_db2)
        # dL_dW1 ~ (x_sz, y1_sz)
        np.dot(self.X.T, self.E1, out=self.dL_dW1)  # dL_dW1 = X.T  @ E1
        # (bias) dL_db1 = E1 summed over the batch
        np.sum(self.E1, axis=0, out=self.dL_db1)

        # fused SGD step: scale grads in place by lr, then subtract (no temp alloc)
        self.dL_dW2 *= lr
        self.dL_dW1 *= lr
        self.dL_db2 *= lr  # (bias)
        self.dL_db1 *= lr  # (bias)
        self.W2 -= self.dL_dW2
        self.W1 -= self.dL_dW1
        self.b2 -= self.dL_db2  # (bias)
        self.b1 -= self.dL_db1  # (bias)

    @staticmethod
    def one_hot(labels, n_classes=10):  # integer labels (batch,) -> (batch, n_classes)
        Y = np.zeros((labels.shape[0], n_classes))
        Y[np.arange(labels.shape[0]), labels] = 1.0
        return Y

    def predict(
        self, X
    ):  # class index with highest probability (expects exactly b_sz rows)
        return self.forward(X).argmax(axis=1)


nn_drfm_b = DigitsRecognizerNetFixedMemB(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2)

W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)
dL_db1: float64 (16,)
dL_db2: float64 (2,)


## canonical PyTorch baseline - DigitsRecognizerPytorch

In [ ]:
# Canonical PyTorch version of the SAME architecture as DigitsRecognizerNetB:
#   Linear(784->128) -> ReLU -> Linear(128->10), classic bias.
# This is the idiomatic-torch counterpart, so it deliberately does NOT mirror our NumPy
# interface: there is no hand-written backward() (autograd computes the gradients) and no
# softmax/one_hot here. forward() returns raw LOGITS -- the canonical choice -- because
# nn.CrossEntropyLoss (used in the training loop below) applies log-softmax internally.
import torch
import torch.nn as nn
import torch.nn.functional as F


class DigitsRecognizerPytorch(nn.Module):
    def __init__(self, *, x_sz, y1_sz, y2_sz):
        super().__init__()
        self.fc1 = nn.Linear(x_sz, y1_sz)  # weight ~ (y1_sz, x_sz), bias ~ (y1_sz,)
        self.fc2 = nn.Linear(y1_sz, y2_sz)  # weight ~ (y2_sz, y1_sz), bias ~ (y2_sz,)

        # match DigitsRecognizerNetB's init: He for the ReLU layer, 1/sqrt(fan_in) for the
        # output layer, zero biases (torch's Linear default differs, so we set it explicitly)
        nn.init.kaiming_normal_(
            self.fc1.weight, nonlinearity="relu"
        )  # std = sqrt(2/x_sz)
        nn.init.normal_(
            self.fc2.weight, std=(1.0 / y1_sz) ** 0.5
        )  # std = sqrt(1/y1_sz)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)

        for (
            name,
            p,
        ) in self.named_parameters():  # print param shapes/dtypes like the NumPy nets
            print(f"{name}: {p.dtype} {tuple(p.shape)}")

    def forward(self, X):  # X ~ (b, x_sz) float tensor -> logits ~ (b, y2_sz)
        return self.fc2(F.relu(self.fc1(X)))


dpt = DigitsRecognizerPytorch(x_sz=784, y1_sz=128, y2_sz=10)

fc1.weight: torch.float32 (128, 784)
fc1.bias: torch.float32 (128,)
fc2.weight: torch.float32 (10, 128)
fc2.bias: torch.float32 (10,)


# verifications

## Gradient checking: numerical vs analytical

**Intuition.** `backward()` computes the *analytical* gradient — a closed-form
$\partial L/\partial \theta$ derived with the chain rule. It's fast but easy to get subtly
wrong (a stray transpose, a missing activation derivative, a flipped sign, a sum over the
wrong axis). A gradient is just a **slope**, and we can measure a slope *independently* and
brute-force by nudging one parameter and watching the loss move. If the hand-derived
analytical gradient matches that measurement, backprop is almost certainly correct.

### The numerical gradient (finite differences)

By definition, the derivative w.r.t. a single parameter $\theta_i$ is

$$\frac{\partial L}{\partial \theta_i} = \lim_{\varepsilon \to 0} \frac{L(\theta_i+\varepsilon) - L(\theta_i-\varepsilon)}{2\varepsilon}.$$

We approximate it for a small $\varepsilon$ with the **central difference**, which is
symmetric and has error $O(\varepsilon^2)$ — far more accurate than the one-sided
forward difference $\big(L(\theta_i+\varepsilon)-L(\theta_i)\big)/\varepsilon$, whose error is only $O(\varepsilon)$:

$$g^{\text{num}}_i = \frac{L(\theta_i+\varepsilon) - L(\theta_i-\varepsilon)}{2\varepsilon}.$$

Each entry costs **two forward passes and no backward pass**. For a weight matrix with
millions of entries that's hopeless for training — but ideal as a *test*, especially if we
only spot-check a **random subset** of entries.

### Comparing the two

Use a scale-free **relative error** rather than a raw difference (a `1e-3` mismatch is fine
for huge gradients, catastrophic for tiny ones):

$$\text{rel} = \frac{|g^{\text{num}}_i - g^{\text{ana}}_i|}{|g^{\text{num}}_i| + |g^{\text{ana}}_i| + \text{tiny}}.$$

For `float64` central differences a correct gradient typically gives $\text{rel} \lesssim 10^{-7}$;
anything above ~$10^{-4}$ signals a bug.

### Pseudocode

```
analytical = backward(...)              # the gradient we want to trust
for i in sampled_parameter_indices:
    theta[i] += eps;   L_plus  = loss(forward())
    theta[i] -= 2*eps; L_minus = loss(forward())
    theta[i] += eps                     # restore exactly
    numerical_i = (L_plus - L_minus) / (2*eps)
    assert rel_err(numerical_i, analytical[i]) < tol
```

### Why *also* "check that it trains"

A gradient check validates **one step at one point**. It can still miss bugs that don't
surface there: a wrong learning-rate sign, an update written to the wrong buffer, stale
state carried across iterations, or exploding/vanishing dynamics. A cheap end-to-end smoke
test — run a few hundred SGD steps and confirm the loss actually goes **down** — catches
those. The two are complementary: the gradient check asks *"is the gradient right?"*, the
train check asks *"does using it actually learn?"*.

In [ ]:
from typing import Literal


def verify_network(
    net,
    *,
    task: Literal["classification", "regression"],
    batch_size=8,
    train_steps=300,
    lr=1e-2,
    n_grad_checks=30,  # random entries sampled per parameter (a full check is too slow)
    eps=1e-6,  # finite-difference step for the numerical gradient
    grad_tol=1e-4,  # max acceptable relative error between numerical & analytical grads
    seed=0,
):
    """Sanity-check an already-constructed network two complementary ways:
      (a) GRADIENT CHECK -- numerical gradient (central finite differences of the loss)
          vs analytical gradient (what backward() actually applies). Catches bugs in the
          backward math (transposes, signs, missing activation derivatives, wrong axes).
      (b) TRAIN CHECK    -- a short SGD loop must drive the loss DOWN. Catches bugs a
          one-point gradient check can miss (wrong lr sign, stale state, bad dynamics).

    Works for every net in this notebook: they share forward(X), backward(Y_true, lr),
    the staticmethod _loss(Y_pred, Y_true), and parameters named W1/W2 (plus b1/b2).
    """
    cls = type(net)
    x_sz = net.W1.shape[0]  # input size  (W1 is (x_sz, y1_sz))
    y2_sz = net.W2.shape[1]  # output size (W2 is (y1_sz, y2_sz))

    # one fixed (X, Y_true) batch for the chosen task
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((batch_size, x_sz))
    if task == "classification":  # one-hot targets so cross-entropy is well defined
        labels = rng.integers(0, y2_sz, size=batch_size)
        Y_true = np.zeros((batch_size, y2_sz))
        Y_true[np.arange(batch_size), labels] = 1.0
    else:  # real-valued targets for the squared-error loss
        Y_true = rng.standard_normal((batch_size, y2_sz))

    loss = lambda: cls._loss(net.forward(X), Y_true)  # scalar loss at current params
    # parameter arrays the net exposes (biases only on the *B variants)
    params = [
        p
        for p in ("W1", "b1", "W2", "b2")
        if isinstance(getattr(net, p, None), np.ndarray)
    ]

    # ---------- (a) GRADIENT CHECK ----------
    # Read the analytical gradient straight from the update: every class does
    #   P <- P - lr * grad,  so ONE step with lr=1 makes the amount each parameter
    # moved equal its gradient (this avoids depending on internal grad-buffer details).
    net.forward(X)
    before = {p: getattr(net, p).copy() for p in params}
    net.backward(Y_true, lr=1.0)
    analytic = {p: before[p] - getattr(net, p) for p in params}
    for p in params:
        getattr(net, p)[:] = before[p]  # put the parameters back

    # numerical gradient on a random subset of entries; compare via relative error
    worst_rel = 0.0
    for p in params:
        flat = getattr(net, p).reshape(
            -1
        )  # view -> writing flat[i] edits the real parameter
        a_flat = analytic[p].reshape(-1)
        for i in rng.choice(
            flat.size, size=min(n_grad_checks, flat.size), replace=False
        ):
            o = flat[i]
            flat[i] = o + eps
            lp = loss()
            flat[i] = o - eps
            lm = loss()
            flat[i] = o  # restore exactly
            num = (lp - lm) / (2 * eps)
            worst_rel = max(
                worst_rel, abs(num - a_flat[i]) / (abs(num) + abs(a_flat[i]) + 1e-12)
            )

    # ---------- (b) TRAIN CHECK ----------
    # the gradient check left the parameters at their initial values, so just train now
    L0 = loss()
    for _ in range(train_steps):
        net.forward(X)
        net.backward(Y_true, lr=lr)
    L1 = loss()

    # ---------- report ----------
    print(f"{cls.__name__}  (params: {params}, task: {task})")
    print(
        f"  (a) gradient check: worst rel err = {worst_rel:.2e}  -> {'PASS' if worst_rel < grad_tol else 'FAIL'}  (tol {grad_tol:.0e})"
    )
    print(
        f"  (b) train check   : loss {L0:.4f} -> {L1:.4f}  -> {'PASS' if L1 < L0 else 'FAIL'}"
    )
    if task == "classification" and hasattr(net, "predict"):
        print(
            f"      train accuracy = {(net.predict(X) == Y_true.argmax(1)).mean():.2f}"
        )


verify_network(Net(x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(NetB(x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(NetFixedMem(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(NetFixedMemB(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2), task="regression")
verify_network(DigitsRecognizerNet(x_sz=5, y1_sz=16, y2_sz=2), task="classification")
verify_network(DigitsRecognizerNetB(x_sz=5, y1_sz=16, y2_sz=2), task="classification")
verify_network(
    DigitsRecognizerNetFixedMemB(b_sz=8, x_sz=5, y1_sz=16, y2_sz=2),
    task="classification",
)

W1: float64 (5, 16)
W2: float64 (16, 2)
Net  (params: ['W1', 'W2'], task: regression)
  (a) gradient check: worst rel err = 7.24e-08  -> PASS  (tol 1e-04)
  (b) train check   : loss 6.0841 -> 0.1704  -> PASS
W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
NetB  (params: ['W1', 'b1', 'W2', 'b2'], task: regression)
  (a) gradient check: worst rel err = 5.86e-09  -> PASS  (tol 1e-04)
  (b) train check   : loss 25.1249 -> 0.0505  -> PASS
W1: float64 (5, 16)
W2: float64 (16, 2)
X: float64 (8, 5)
Z1: float64 (8, 16)
Y1: float64 (8, 16)
Z2: float64 (8, 2)
Y2: float64 (8, 2)
E2: float64 (8, 2)
E1: float64 (8, 16)
D1: float64 (8, 16)
D2: float64 (8, 2)
dL_dW1: float64 (5, 16)
dL_dW2: float64 (16, 2)
NetFixedMem  (params: ['W1', 'W2'], task: regression)
  (a) gradient check: worst rel err = 4.11e-09  -> PASS  (tol 1e-04)
  (b) train check   : loss 63.9206 -> 0.0411  -> PASS
W1: float64 (5, 16)
b1: float64 (16,)
W2: float64 (16, 2)
b2: float64 (2,)
X: float64 (8, 5)
Z1:

In [ ]:
# Torch counterpart of verify_network. Same two checks, but the ANALYTICAL gradient now
# comes from autograd (loss.backward() -> p.grad) instead of a hand-written backward:
#   (a) gradient check: autograd grads vs central finite differences of the loss.
#   (b) train check   : a short SGD loop must drive the loss DOWN.
# Assumes a classification model: forward(X) -> logits, trained with cross-entropy.
# NOTE: we run the check in float64 (model.double()). Finite differences subtract two
# nearly-equal losses, so float32 round-off would swamp the signal -- this is exactly why
# torch's built-in torch.autograd.gradcheck also requires double precision.
def verify_network_torch(
    model,
    *,
    batch_size=8,
    train_steps=300,
    lr=1e-2,
    n_grad_checks=30,  # random entries sampled per parameter (a full check is too slow)
    eps=1e-6,  # finite-difference step for the numerical gradient
    grad_tol=1e-4,  # max acceptable relative error between numerical & analytical grads
    seed=0,
):
    model = (
        model.double()
    )  # float64 so finite differences are accurate (see note above)
    x_sz = model.fc1.weight.shape[1]  # in_features
    y2_sz = model.fc2.weight.shape[0]  # out_features == number of classes

    # one fixed (X, targets) batch; CrossEntropyLoss wants integer class labels
    g = torch.Generator().manual_seed(seed)
    X = torch.randn(batch_size, x_sz, dtype=torch.float64, generator=g)
    targets = torch.randint(0, y2_sz, (batch_size,), generator=g)
    criterion = nn.CrossEntropyLoss()
    loss = lambda: criterion(model(X), targets)  # scalar loss at current params

    # ---------- (a) GRADIENT CHECK ----------
    # analytical gradient: a single backward pass fills p.grad for every parameter
    model.zero_grad()
    loss().backward()
    analytic = [p.grad.clone() for p in model.parameters()]

    # numerical gradient on a random subset of entries; compare via relative error
    worst_rel = 0.0
    with torch.no_grad():  # we only read loss values here, no graph needed
        for p, a in zip(model.parameters(), analytic):
            flat = p.view(-1)  # view -> writing flat[i] edits the real parameter
            a_flat = a.view(-1)
            idx = torch.randperm(flat.numel(), generator=g)[
                : min(n_grad_checks, flat.numel())
            ]
            for i in idx:
                o = flat[i].item()
                flat[i] = o + eps
                lp = loss().item()
                flat[i] = o - eps
                lm = loss().item()
                flat[i] = o  # restore exactly
                num = (lp - lm) / (2 * eps)
                worst_rel = max(
                    worst_rel,
                    abs(num - a_flat[i].item())
                    / (abs(num) + abs(a_flat[i].item()) + 1e-12),
                )

    # ---------- (b) TRAIN CHECK ----------
    # the gradient check left the parameters at their initial values, so just train now
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    L0 = loss().item()
    for _ in range(train_steps):
        optimizer.zero_grad()
        loss().backward()
        optimizer.step()
    L1 = loss().item()

    # ---------- report ----------
    names = [n for n, _ in model.named_parameters()]
    print(f"{type(model).__name__}  (params: {names}, task: classification)")
    print(
        f"  (a) gradient check: worst rel err = {worst_rel:.2e}  -> {'PASS' if worst_rel < grad_tol else 'FAIL'}  (tol {grad_tol:.0e})"
    )
    print(
        f"  (b) train check   : loss {L0:.4f} -> {L1:.4f}  -> {'PASS' if L1 < L0 else 'FAIL'}"
    )


verify_network_torch(DigitsRecognizerPytorch(x_sz=5, y1_sz=16, y2_sz=2))

fc1.weight: torch.float32 (16, 5)
fc1.bias: torch.float32 (16,)
fc2.weight: torch.float32 (2, 16)
fc2.bias: torch.float32 (2,)
DigitsRecognizerPytorch  (params: ['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'], task: classification)
  (a) gradient check: worst rel err = 4.46e-08  -> PASS  (tol 1e-04)
  (b) train check   : loss 0.7450 -> 0.2469  -> PASS


In [ ]:
# Same (a) gradient check, but using torch's BUILT-IN tool instead of our hand-rolled loop.
# torch.autograd.gradcheck compares autograd's analytical Jacobian against numerical finite
# differences (in float64) and raises if they disagree -- exactly the (a) check, batteries
# included. It checks gradients w.r.t. the TENSORS you pass it, so to verify PARAMETER
# gradients we express the loss as a function of the parameters with torch.func.functional_call
# (it runs the model but with its parameters swapped for the tensors we hand in).
from torch.autograd import gradcheck
from torch.func import functional_call

torch.manual_seed(0)
model = DigitsRecognizerPytorch(
    x_sz=5, y1_sz=16, y2_sz=2
).double()  # double -> accurate finite diffs

X = torch.randn(8, 5, dtype=torch.float64)  # (b, x_sz)
targets = torch.randint(0, 2, (8,))  # (b,) integer class labels
criterion = nn.CrossEntropyLoss()

names = [n for n, _ in model.named_parameters()]  # parameter names, in order
params = tuple(
    p.detach().clone().requires_grad_(True) for p in model.parameters()
)  # the inputs to check


def loss_from_params(
    *params,
):  # rebuild the forward pass from the supplied params -> scalar loss
    logits = functional_call(model, dict(zip(names, params)), (X,))  # (b, 2)
    return criterion(logits, targets)


# returns True if every analytical grad matches finite differences within tolerance (else raises)
ok = gradcheck(loss_from_params, params, eps=1e-6, atol=1e-4)
print(f"torch.autograd.gradcheck passed: {ok}")

fc1.weight: torch.float32 (16, 5)
fc1.bias: torch.float32 (16,)
fc2.weight: torch.float32 (2, 16)
fc2.bias: torch.float32 (2,)
torch.autograd.gradcheck passed: True


# task example: digits recognizing

In [81]:
# End-to-end MNIST test for our classifier nets. `net_cls` is a parameter, so the SAME
# routine works for any class with the DigitsRecognizer* interface
# (forward / backward / one_hot / predict) -- swap the class at the call site below.
#
# torchvision is used ONLY to download the dataset (the exact same bits a PyTorch
# baseline would see, for a fair comparison); ALL training/inference is our NumPy code.
from torchvision import datasets


def load_mnist():
    """Download MNIST and return flat, normalized NumPy arrays.
    torchvision hands us:
      ds.data    : uint8  torch tensor (N, 28, 28)  -- raw pixel intensities 0..255
      ds.targets : int64  torch tensor (N,)         -- digit labels 0..9
    """
    train = datasets.MNIST(root="./data", train=True, download=True)
    test = datasets.MNIST(root="./data", train=False, download=True)
    print(
        f"raw train images: {tuple(train.data.shape)} {train.data.dtype}"
    )  # (60000, 28, 28) uint8
    print(
        f"raw train labels: {tuple(train.targets.shape)} {train.targets.dtype}"
    )  # (60000,) int64

    # canonical MNIST mean/std (what torchvision's transforms.Normalize uses) so our
    # inputs match a standard PyTorch pipeline; helps the He-initialized first layer.
    MEAN, STD = 0.1307, 0.3081

    def to_xy(ds):
        # (N, 28, 28) uint8 -> (N, 784) float64, scaled to [0,1] then standardized
        X = ds.data.numpy().reshape(ds.data.shape[0], -1).astype(np.float64) / 255.0
        X = (X - MEAN) / STD
        y = ds.targets.numpy().astype(np.int64)  # (N,)
        return X, y

    X_train, y_train = to_xy(train)
    X_test, y_test = to_xy(test)
    print(
        f"X_train: {X_train.shape} {X_train.dtype}   y_train: {y_train.shape} {y_train.dtype}"
    )
    print(
        f"X_test : {X_test.shape} {X_test.dtype}   y_test : {y_test.shape} {y_test.dtype}"
    )
    return X_train, y_train, X_test, y_test


def accuracy(net, X, y, batch_size=1000):
    # predict in chunks (keeps intermediates small), then compare to the int labels
    preds = np.concatenate(
        [net.predict(X[i : i + batch_size]) for i in range(0, X.shape[0], batch_size)]
    )
    return (preds == y).mean()  # preds ~ (N,) int  ==  y ~ (N,) int


def train_and_test_mnist(
    net_cls, *, y1_sz=128, epochs=3, batch_size=128, lr=0.1, seed=0
):
    """Train `net_cls` on MNIST with mini-batch SGD, printing shapes + per-epoch accuracy."""
    X_train, y_train, X_test, y_test = load_mnist()
    n, x_sz = X_train.shape  # 60000, 784
    n_classes = 10

    np.random.seed(seed)  # reproducible weight init
    net = net_cls(
        x_sz=x_sz, y1_sz=y1_sz, y2_sz=n_classes
    )  # __init__ prints param shapes

    rng = np.random.default_rng(seed)
    for epoch in range(epochs):
        order = rng.permutation(n)  # reshuffle the dataset each epoch
        for i in range(0, n, batch_size):
            idx = order[i : i + batch_size]
            Xb = X_train[idx]  # (b, 784) float64
            Yb = net.one_hot(y_train[idx], n_classes)  # (b, 10) one-hot float64
            if epoch == 0 and i == 0:  # peek at the very first mini-batch
                print(
                    f"batch X: {Xb.shape} {Xb.dtype}   batch Y(one-hot): {Yb.shape} {Yb.dtype}"
                )
            net.forward(Xb)
            net.backward(Yb, lr=lr)
        print(
            f"epoch {epoch + 1}/{epochs}: train acc = {accuracy(net, X_train, y_train):.4f}   test acc = {accuracy(net, X_test, y_test):.4f}"
        )

    print(f"predict sample: {net.predict(X_test[:10])}  (true: {y_test[:10]})")
    return net


net_mnist = train_and_test_mnist(DigitsRecognizerNetB)

raw train images: (60000, 28, 28) torch.uint8
raw train labels: (60000,) torch.int64
X_train: (60000, 784) float64   y_train: (60000,) int64
X_test : (10000, 784) float64   y_test : (10000,) int64
W1: float64 (784, 128)
b1: float64 (128,)
W2: float64 (128, 10)
b2: float64 (10,)
batch X: (128, 784) float64   batch Y(one-hot): (128, 10) float64
epoch 1/3: train acc = 0.9525   test acc = 0.9482
epoch 2/3: train acc = 0.9636   test acc = 0.9581
epoch 3/3: train acc = 0.9755   test acc = 0.9685
predict sample: [7 2 1 0 4 1 4 9 6 9]  (true: [7 2 1 0 4 1 4 9 5 9])


In [ ]:
# Idiomatic PyTorch training loop for DigitsRecognizerPytorch -- the canonical counterpart
# to train_and_test_mnist above. Same data (reuses load_mnist) and same plain SGD, so the
# numbers are comparable; the mechanics are the textbook torch ones, with autograd doing the
# backward pass we hand-wrote in the NumPy nets:
#   optimizer.zero_grad() -> logits = model(x) -> loss = criterion(logits, y)
#   -> loss.backward()  (autograd)  -> optimizer.step().
from torch.utils.data import DataLoader, TensorDataset


def train_and_test_mnist_torch(*, y1_sz=128, epochs=3, batch_size=128, lr=0.1, seed=0):
    X_train, y_train, X_test, y_test = load_mnist()

    # numpy -> tensors (float32 inputs, int64 class labels -- what CrossEntropyLoss wants)
    Xtr = torch.tensor(X_train, dtype=torch.float32)  # (60000, 784)
    ytr = torch.tensor(y_train, dtype=torch.long)  # (60000,)
    Xte = torch.tensor(X_test, dtype=torch.float32)  # (10000, 784)
    yte = torch.tensor(y_test, dtype=torch.long)  # (10000,)
    # DataLoader handles per-epoch shuffling + batching (the canonical torch input pipeline)
    train_loader = DataLoader(
        TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True
    )

    torch.manual_seed(seed)  # reproducible weight init
    model = DigitsRecognizerPytorch(x_sz=Xtr.shape[1], y1_sz=y1_sz, y2_sz=10)
    criterion = (
        nn.CrossEntropyLoss()
    )  # softmax + cross-entropy, averaged over the batch
    optimizer = torch.optim.SGD(
        model.parameters(), lr=lr
    )  # plain SGD, like our NumPy nets

    @torch.no_grad()
    def accuracy(X, y):
        model.eval()  # eval mode (no effect here; matters once you add dropout/batchnorm)
        return (
            (model(X).argmax(dim=1) == y).float().mean().item()
        )  # logits -> class -> mean hits

    Xb0, yb0 = next(iter(train_loader))  # peek at one mini-batch
    print(
        f"batch X: {tuple(Xb0.shape)} {Xb0.dtype}   batch y: {tuple(yb0.shape)} {yb0.dtype}"
    )
    for epoch in range(epochs):
        model.train()  # train mode
        for Xb, yb in train_loader:  # Xb ~ (b, 784), yb ~ (b,) int64
            optimizer.zero_grad()  # clear grads from the previous step
            loss = criterion(model(Xb), yb)  # forward pass -> scalar loss
            loss.backward()  # autograd fills every parameter's .grad
            optimizer.step()  # SGD update: p -= lr * p.grad
        print(
            f"epoch {epoch + 1}/{epochs}: train acc = {accuracy(Xtr, ytr):.4f}   test acc = {accuracy(Xte, yte):.4f}"
        )

    with torch.no_grad():
        sample = model(Xte[:10]).argmax(dim=1)
    print(f"predict sample: {sample.tolist()}  (true: {yte[:10].tolist()})")
    return model


net_mnist_torch = train_and_test_mnist_torch()

raw train images: (60000, 28, 28) torch.uint8
raw train labels: (60000,) torch.int64
X_train: (60000, 784) float64   y_train: (60000,) int64
X_test : (10000, 784) float64   y_test : (10000,) int64
fc1.weight: torch.float32 (128, 784)
fc1.bias: torch.float32 (128,)
fc2.weight: torch.float32 (10, 128)
fc2.bias: torch.float32 (10,)
batch X: (128, 784) torch.float32   batch y: (128,) torch.int64
epoch 1/3: train acc = 0.9513   test acc = 0.9496
epoch 2/3: train acc = 0.9654   test acc = 0.9602
epoch 3/3: train acc = 0.9764   test acc = 0.9704
predict sample: [7, 2, 1, 0, 4, 1, 4, 9, 6, 9]  (true: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9])


# [EOF]